In [ ]:
import sys
sys.path.append('../Model_info')
import os
import Stdinput as orctm
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime,timedelta
import time

In [ ]:
INI=orctm.Orctm_cvcm('ORCTM_cvcm','Surface fields for Temperature')
latp_grid = INI.model_info_p('latp_grid')
lonp_grid = INI.model_info_p('lonp_grid')
#zzout = INI.model_info_p('zzout')
mask = INI.model_info_p('weto_p')
mask = mask.astype(int)

In [ ]:
#cmems_file = './2011/CMEMS-For-CVCM_SUR_file_2011.nc'
#cmems_file = './2012/CMEMS-For-CVCM_SUR_file_2012.nc'
#cmems_file = './2013/CMEMS-For-CVCM_SUR_file_2013.nc'
#cmems_file = './2014/CMEMS-For-CVCM_SUR_file_2014.nc'
#cmems_file = './2015/CMEMS-For-CVCM_SUR_file_2015.nc'
cmems_file = './2016/CMEMS-For-CVCM_SUR_file_2016.nc'
ini_file = nc.Dataset(cmems_file,mode='r',format='netCDF4')
#print(ini_file.variables)
latitude = ini_file.variables['latitude'][:]
longitude = ini_file.variables['longitude'][:]
longitude_grid,latitude_grid = np.meshgrid(longitude,latitude)
DEP = ini_file.variables['depth'][:]
so = ini_file.variables['so'][:,:,:,:]
so = so.squeeze()
so = so.filled(fill_value = np.nan)
nc.Dataset.close(ini_file)


In [ ]:
start_time = time.time()
# Start the timer
te = 12
so_sss = np.zeros((INI.ie_g,INI.je_g))
Lat_grid = INI.rearrange_grid_to_orctm(latitude_grid)
Lon_grid = INI.rearrange_grid_to_orctm(longitude_grid)

# 
import struct
if22 = 2 # Salinity for SSS

with open('SURSAL', 'wb') as sursal:
    for t in range(te):
        print('Month: ',str(t+1),' ')
        so_grid  = INI.rearrange_grid_to_orctm(so[t,:,:])
        land = np.zeros_like(so_grid)
        land[np.isnan(so_grid)]=1  # land 1 Sea: 0 !!!!
        if np.sum(land) != np.size(land):
            so_sss = INI.Grid_bilinear_interpolation( \
                so_grid,land,Lat_grid,Lon_grid,latp_grid,lonp_grid,landvalue=np.nanmean(so_grid))
       
        so_sss[mask[:,:,0]==0] = np.nan
        # check obc
        # minimum
        so_sss = np.where(so_sss <= np.nanmin(so_grid),np.nanmin(so_grid),so_sss)
        # maximum
        so_sss = np.where(so_sss >= np.nanmax(so_grid),np.nanmax(so_grid),so_sss)
        so_sss[np.isnan(so_sss)==True]=0
        
        header = struct.pack('>qqqq', int(t+1), int(if22), int(INI.ZZOUT[0]), int(INI.ie_g * INI.je_g))
        header_len = len(header)

        # Wrire the Head info
        # >i
        sursal.write(struct.pack(">i", header_len))  # length of header
        sursal.write(header)                         # DATA length
        sursal.write(struct.pack(">i", header_len))  # length of tail

        # Data arrays 
        sal_sss = so_sss.astype('>f8')  #  double precision float
        sal_sss = sal_sss.T  
        sal_data = sal_sss.tobytes()  #  endian 64-bit float
        sal_len = len(sal_data)

        # Write the INITEM float data
        sursal.write(struct.pack(">i", sal_len))  # length of previous record
        sursal.write(sal_data)                    # DATA block
        sursal.write(struct.pack(">i", sal_len))  # length of tail

# Print elapsed time
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.6f} seconds")